# 09 — Atlas AI: the agent your model was for

Chapter 15. Point an agent at the model you built, ask it questions in English, and see
why each Chapter 03 decision is what makes the answer possible.

**The agents API is alpha.** The SDK prints a FeaturePreviewWarning on first use; that is
the contract, not noise.

## Setup — identical to notebook 07

In [ ]:
# ---------------------------------------------------------------- setup ----
import os
from pathlib import Path

from cognite.client import CogniteClient, global_config
global_config.disable_pypi_version_check = True
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, OAuthInteractive

# Find the repo root by its markers, so this cell works wherever Jupyter started.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "pyproject.toml").exists() and (p / "training").exists()), HERE)

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        if " #" in v and not v.startswith(('"', "'")):
            v = v.split(" #", 1)[0].rstrip()
        os.environ.setdefault(k, v)      # a real environment variable always wins

missing = [k for k in ("CDF_PROJECT", "CDF_CLUSTER", "IDP_CLIENT_ID")
           if not os.environ.get(k)]
assert not missing, f"Missing {missing}. Copy .env.example to {env_path} and fill it in."


def cdf_client(name: str) -> CogniteClient:
    """Build the client EXPLICITLY.

    `CogniteClient()` with no arguments does not read your .env. The SDK removed
    implicit construction in v8 and raises:
        ValueError: No ClientConfig has been provided
    The branch below is the two-identity rule from Chapter 02, in code.
    """
    base_url = os.environ.get("CDF_URL") or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
    scopes = [s for s in os.environ.get("IDP_SCOPES", f"{base_url}/.default").split(",") if s]

    if os.environ.get("LOGIN_FLOW", "interactive").lower() == "interactive":
        creds = OAuthInteractive(              # you, in a browser -- needs
            authority_url=os.environ["IDP_AUTHORITY_URL"],   # localhost:53000
            client_id=os.environ["IDP_CLIENT_ID"],           # as a redirect URI
            scopes=scopes)
    else:
        creds = OAuthClientCredentials(        # unattended: a service principal
            token_url=os.environ["IDP_TOKEN_URL"],
            client_id=os.environ["IDP_CLIENT_ID"],
            client_secret=os.environ["IDP_CLIENT_SECRET"],
            scopes=scopes)

    return CogniteClient(ClientConfig(
        client_name=name, project=os.environ["CDF_PROJECT"],
        base_url=base_url, credentials=creds))


YOURNAME = os.environ.get("PARTICIPANT", "YOURNAME")   # [CHANGE] if not in .env
client   = cdf_client(f"dm-handson-{YOURNAME}-query")

space       = f"isp_{YOURNAME}_TRN"
schema_edm  = f"ssp_{YOURNAME}_TrainingCore_edm"
schema_sdm  = f"ssp_{YOURNAME}_MaintenanceInsight_sdm"
raw_db      = f"rwd_{YOURNAME}_Training_TRN"
model_version = "v1.0.0"


# --- identifiers every chapter uses ---------------------------------------
from cognite.client.data_classes.data_modeling import ViewId
from cognite.client.data_classes import filters as flt
from cognite.client.data_classes.data_modeling.query import (
    Query, QuerySync, NodeResultSetExpression, EdgeResultSetExpression,
    Select, SourceSelector)
from cognite.client.data_classes.data_modeling import (
    NodeId, EdgeId, NodeApply, EdgeApply, NodeOrEdgeData, DirectRelationReference)
from cognite.client.data_classes.raw import Row

from cognite.client.data_classes.aggregations import Count, Avg, Max

INSTANCE_SPACE = space
EDM_SPACE      = schema_edm
SDM_SPACE      = schema_sdm
RAW_DB         = raw_db
MODEL_VERSION  = model_version

ASSET      = ViewId("cdf_cdm", "CogniteAsset",     "v1")
EQUIPMENT  = ViewId("cdf_cdm", "CogniteEquipment", "v1")
ACTIVITY   = ViewId("cdf_cdm", "CogniteActivity",  "v1")
TIMESERIES = ViewId("cdf_cdm", "CogniteTimeSeries","v1")
FILE       = ViewId("cdf_cdm", "CogniteFile",      "v1")
WORKORDER  = ViewId(EDM_SPACE, "WorkOrder",              MODEL_VERSION)
EHP        = ViewId(SDM_SPACE, "EquipmentHealthProfile", MODEL_VERSION)

print("connected:", client.config.project, "| space:", space)

## section 15.3 — create the agent, scoped to your model and your space

In [ ]:
from cognite.client.data_classes.agents import (
    AgentUpsert, QueryKnowledgeGraphAgentToolUpsert,
    QueryKnowledgeGraphAgentToolConfiguration, QueryTimeSeriesDatapointsAgentToolUpsert,
    DataModelInfo, InstanceSpaces, Message,
)

AGENT_XID = f"agt_{YOURNAME}_maintenance"

tool = QueryKnowledgeGraphAgentToolUpsert(
    name="maintenance_graph",
    # This description is how the agent decides whether to use the tool at all.
    # "The maintenance graph" would not be enough.
    description=(
        "The MaintenanceInsight model for one FPSO separation train: assets, equipment, "
        "work orders, equipment health profiles with datasheet specs, P&ID annotations "
        "and time series."
    ),
    configuration=QueryKnowledgeGraphAgentToolConfiguration(
        data_models=[DataModelInfo(
            space=SDM_SPACE, external_id="MaintenanceInsight", version=MODEL_VERSION,
            view_external_ids=[
                "Asset", "EquipmentHealthProfile", "WorkOrder", "CogniteEquipment",
                "CogniteTimeSeries", "CogniteFile", "CogniteDiagramAnnotation",
            ],
        )],
        instance_spaces=InstanceSpaces(type="manual", spaces=[INSTANCE_SPACE]),
    ),
)

datapoints_tool = QueryTimeSeriesDatapointsAgentToolUpsert(
    name="sensor_history",
    description=(
        "Retrieve time-series datapoints after maintenance_graph has identified the "
        "relevant CogniteTimeSeries nodes. Use it for trends and measured values; "
        "never infer an engineering limit that the graph does not contain."
    ),
)

agent = client.agents.upsert(AgentUpsert(
    external_id=AGENT_XID,
    name=f"{YOURNAME} Maintenance Insight",
    description="Answers reliability questions about the separation train.",
    instructions=(
        "You answer maintenance and reliability questions about one FPSO separation "
        "train. Always cite the externalIds of the instances you used. Units are "
        "declared on the properties -- state them. If the configured tools do not contain "
        "the answer, say so plainly instead of guessing."
    ),
    tools=[tool, datapoints_tool],
))
print(agent.external_id, "->", [t.name for t in agent.tools])

## section 15.4 — ask it what you already verified by hand

In [ ]:
def ask(question: str) -> str:
    response = client.agents.chat(
        agent_external_id=AGENT_XID, messages=Message(content=question))
    print(f"Q: {question}\n\nA: {response.text}\n")
    return response.text

# You computed this answer by hand in Chapter 13 section 13.5: WO-1001, IN_PROGRESS, 18500 EUR.
a1 = ask("Which work orders are not closed against pump 21-PA-2001A, and what do they cost?")

In [ ]:
# Only answerable because of three schema decisions:
#   1. Asset.healthProfile  -- the reverse direct relation (Ch03 section 3.12)
#   2. property descriptions on EquipmentHealthProfile (Ch03 section 3.11)
#   3. source: on datasheetFile, so the file resolves to a view (Ch10)
a2 = ask("What is the rated power and seal type of pump 21-PA-2001A, "
         "and which document did that come from?")

In [ ]:
# Three ingestion techniques, three chapters, one decision-grade question.
# A trustworthy answer separates measured evidence, a recorded maintenance response,
# and nameplate context. The graph contains no vibration alarm threshold or confirmed root cause.
a3 = ask("Pump 21-PA-2001A: what evidence suggests degraded performance, "
         "what maintenance response is underway, and what does its datasheet tell us?")

## section 15.5 — break it on purpose\n\nAn agent that invents an answer here is worse than useless.

In [ ]:
a4 = ask("Who is the manufacturer's service representative for pump 21-PA-2001A?")

refused = any(w in a4.lower() for w in
              ("not", "no ", "does not", "unable", "cannot", "don't", "unknown"))
print("declined rather than invented:", refused)

## section 15.7 — clean up\n\nThe agent is global. Purging your spaces does not remove it.

In [ ]:
client.agents.delete(AGENT_XID, ignore_unknown_ids=True)
print([a.external_id for a in client.agents.list() if YOURNAME in (a.external_id or "")])